# Importing modules and settings

### Importing Libraries

In [ ]:
import numpy as np
import pandas  as pd
import scanpy as sc
import matplotlib.pyplot as plt
from matplotlib.pyplot import rc_context
import seaborn as sns
import os

In [ ]:
import gzip
import fnmatch
import re
from scipy.sparse import csr_matrix
import anndata as ad

### General settings of Scanpy

In [ ]:
sc.settings.figdir = './figures_250523/'

In [ ]:
sc.settings.verbosity = 4
sc.logging.print_header()
sc.settings.set_figure_params(dpi=80, facecolor='white')

In [ ]:
# Create a CMAP for the UMAP plotting
umap_cmap = sns.blend_palette(['xkcd:light grey', 'xkcd:indigo'], as_cmap = True)

In [ ]:
# Declare the output file
name_of_analysis = 'Smed_L78-L47_20250523'
results_file = name_of_analysis + '_Results.h5ad'
results_file

# input file

In [ ]:
adata = sc.read_h5ad( name_of_analysis + '_doublets.h5ad')

In [ ]:
adata

## filter out the doublets

In [ ]:
# set predicted_doublets as a boolean
li = [eval(i) for i in list(adata.obs['predicted_doublet'])]
adata.obs['predicted_doublet'] = li

In [ ]:
# remove the doublets from adata
adata = adata[adata.obs['predicted_doublet'] == False]

In [ ]:
adata.obs

# Quality control and filtering

In [ ]:
# Visualise the expression levels of the top expressed genes in the dataset
sc.pl.highest_expr_genes(adata, n_top=20, ) 

In [ ]:
# Cell numbers per sample in the unfiltered object
sns.countplot(data=adata.obs, x='Sample', hue = 'Sample', palette = list(adata.uns['Sample_colors']))
plt.xticks(rotation=90)
plt.show()

In [ ]:
# Filter by minimum counts and by minimum genes
sc.pp.filter_cells(adata, min_counts= 150)
sc.pp.filter_cells(adata, min_genes= 150)

In [ ]:
# Visualise the top 20 expressed genes
sc.pl.highest_expr_genes(adata, n_top=20, )

In [ ]:
adata.var.columns

In [ ]:
sc.pp.calculate_qc_metrics(adata, percent_top=None, log1p=False, inplace=True)

In [ ]:
adata.var.columns

In [ ]:
# n_genes and n_counts per experiment
sc.pl.violin(adata, ['n_genes_by_counts', 'total_counts'], groupby = 'Experiment', 
             jitter=False, multi_panel=True, log = True)

In [ ]:
adata.obs

In [ ]:
# n_genes and n_counts per sublibrary
sc.pl.violin(adata, ['n_genes_by_counts', 'total_counts'], groupby = 'Sublibrary', 
             jitter=False, multi_panel=True, log = True, rotation = 90)

In [ ]:
# n_genes and n_counts per sample
sc.pl.violin(adata, ['n_genes_by_counts', 'total_counts'], groupby = 'Sample', 
             jitter=False, multi_panel=True, log = True, rotation = 90)

In [ ]:
# Cell numbers per sample in the filtered object
sns.countplot(data=adata.obs, x='Sample', hue = 'Sample', palette = list(adata.uns['Sample_colors']))
plt.xticks(rotation=90)
plt.show()

In [ ]:
adata

# Matrix slicing

In [ ]:
# Scatter plot of the correlation between total_counts and n_genes_by_count
sc.pl.scatter(adata, x='total_counts', y='n_genes_by_counts')

In [ ]:
# Filter by max_counts and max_genes
sc.pp.filter_cells(adata, max_counts=1500)
sc.pp.filter_cells(adata, max_genes= 1200)

In [ ]:
sc.pl.scatter(adata, x='total_counts', y='n_genes_by_counts')

In [ ]:
samp = 'Sample'

In [ ]:
# check for batch effects by visualising the samples
sc.pl.scatter(adata, x='total_counts', y='n_genes_by_counts', color = samp)

In [ ]:
sc.pl.scatter(adata, x='total_counts', y='n_genes_by_counts', color = 'Experiment', groups = 'FACS')

In [ ]:
sc.pl.scatter(adata, x='total_counts', y='n_genes_by_counts', color = 'Experiment', groups = 'RNAi')

In [ ]:
# Saving an unprocessed h5ad object for pseudobulk analyses later
adata.write(name_of_analysis + '_unprocessed.h5ad')

# Matrix normalisation

In [ ]:
# Normalise the data
sc.pp.normalize_total(adata) 

In [ ]:
# log transform avoiding the log of 0
sc.pp.log1p(adata) 

# Selecting highly variable genes

In [ ]:
adata.var.columns

In [ ]:
# Identify the top 20000 highly variable genes.
sc.pp.highly_variable_genes(adata, n_top_genes = 20000) 

In [ ]:
# Visualise the gene distribution
sc.pl.highly_variable_genes(adata)

In [ ]:
adata.var.columns

# Creating an adata.raw object

In [ ]:
# Store complete matrix
adata.raw = adata

In [ ]:
# Create an adata object that contains only the highly variable genes
adata = adata[:, adata.var.highly_variable]

In [ ]:
adata

In [ ]:
adata.raw.var

# Scaling the data

In [ ]:
# Perform data scaling with z-score standardisation
sc.pp.scale(adata)

# Performing the PCA and kNN analysis

In [ ]:
# Perform PCA analysis
sc.tl.pca(adata, svd_solver='arpack', n_comps = 150)

In [ ]:
# Plot the proportion of variance explained by each PC
sc.pl.pca_variance_ratio(adata, n_pcs=150, log=True)

In [ ]:
# integration of the 2 experiments with harmony to mitigate batch effect
sc.external.pp.harmony_integrate(adata, 'Experiment', basis='X_pca')

In [ ]:
# Build the kNN tree
sc.pp.neighbors(adata, n_neighbors=30, n_pcs=110, use_rep='X_pca_harmony')

In [ ]:
# Compute UMAP and plot
sc.tl.umap(adata, min_dist= 0.75, spread = 1.25, alpha = 1, gamma = 2)

In [ ]:
sc.pl.umap(adata)

In [ ]:
# UMAP with expression of Smedwi1
with plt.rc_context({'figure.figsize': (10, 10)}):
    sc.pl.umap(adata, color='h1SMcG0013999', size = 30, color_map = umap_cmap, save = '_smedwi1')

In [ ]:
# check the integration of the 2 datasets
with plt.rc_context({'figure.figsize': (10, 10)}):
    sc.pl.umap(adata, color='Experiment', size = 20, color_map = umap_cmap, save = '_batch.png')

In [ ]:
# umap by conditions
with plt.rc_context({'figure.figsize': (10, 10)}):
    sc.pl.umap(adata, color='Condition', size = 20, color_map = umap_cmap, save = '_condition.png' )

In [ ]:
# G1-G2 from L47 and GFP from L78 should have a similar distribution
with plt.rc_context({'figure.figsize': (10, 10)}):
    sc.pl.umap(adata, color='Condition', groups = ['G1-G2', 'GFP'], size = 20, color_map = umap_cmap)

In [ ]:
# check the doublet score
with plt.rc_context({'figure.figsize': (10, 10)}):
    sc.pl.umap(adata, color='doublet_score', groups = ['G1-G2'], size = 20, color_map = umap_cmap)

# Markers

In [ ]:
# Upload the markers from the Sizes dataset (Emili et al bioRxiv 2023)
markers = pd.read_excel("annotated_cluster_markers_20240329.xlsx")

In [ ]:
markers

In [ ]:
# Extract the markers from the diagnostic name column and add them to a list
marker_list = markers['Diagnostic name marker'].to_list() 

In [ ]:
# Convert the 'Cluster name' column into a list
name_list = markers['Cluster name'].to_list()

In [ ]:
# Create a matplotlib figure
fig, axs = plt.subplots(8, 6, figsize=(40, 50))

# Plot UMAP for each marker
for i, marker in enumerate(marker_list):
    row = i // 6
    col = i % 6
    sc.pl.umap(adata, color=marker, color_map = umap_cmap, size=30, title = name_list[i], ax=axs[row, col], show=False)

plt.tight_layout()
plt.savefig("umap_markers.pdf")
plt.show()
plt.close()   

# Clustering

In [ ]:
# Select resolutions 
resolutions = [1, 1.5, 2, 2.5, 3]

In [ ]:
# Run Leiden for each of the above resolution
for i in resolutions:
    sc.tl.leiden(adata, resolution = i, key_added = 'leiden_'+str(i))
    sc.pl.umap(adata, color='leiden_'+str(i))

In [ ]:
leiden_names = adata.obs.columns[adata.obs.columns.str.contains('leiden')].to_list()

In [ ]:
leiden_names

In [ ]:
# Plot the clusters with the labels on top of the data, for each Leiden resolution
sc.pl.umap(adata, color=leiden_names, legend_loc = 'on data', legend_fontsize = 10)

In [ ]:
# Save the plots
for leiden_i in leiden_names:
    with plt.rc_context({'figure.figsize': (15, 15)}):
        sc.pl.umap(adata, color=leiden_i, legend_loc='on data', title=str(leiden_i), size = 50, frameon=False,
                   save = '_'+str(leiden_i))

In [ ]:
# Compute a correlation matrix to group the Leiden clusters into broad groups
for leiden_i in leiden_names:
    sc.pl.correlation_matrix(adata, leiden_i, figsize=(25,25),
                            save = '_'+str(leiden_i))


In [ ]:
# Generate a dendogram to group the Leiden clusters into broad groups 
for leiden_i in leiden_names:
    with plt.rc_context({'figure.figsize': (15, 5)}):
        sc.pl.dendrogram(adata, leiden_i,
                        save = '_'+str(leiden_i))

In [ ]:
# Find markers using the logreg method
for leiden_i in leiden_names:
    sc.tl.rank_genes_groups(adata, leiden_i, method='logreg', key_added = 'rank_genes_groups_logreg_'+str(leiden_i))
    sc.pl.rank_genes_groups(adata, key='rank_genes_groups_logreg_'+str(leiden_i), n_genes = 10, sharey = False)

In [ ]:
# Find markers using the wilcoxon method
for leiden_i in leiden_names:
    sc.tl.rank_genes_groups(adata, leiden_i, method='wilcoxon', key_added = 'rank_genes_groups_wilcox_'+str(leiden_i))
    sc.pl.rank_genes_groups(adata, key='rank_genes_groups_wilcox_'+str(leiden_i), n_genes = 10, sharey = False)

# Output files

In [ ]:
adata.write(results_file)